Reading from Bronze

INIT


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, size, split
from pyspark.sql.functions import year, month, dayofmonth
from pyspark.sql.functions import to_date
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
def log(step, message):
    print(f"[{step}] {message}")

Load Bronze

In [0]:
log("Silver", "Loading bronze data")

df = spark.table("workspace.bronze.sales_details")

log("Silver", f"Initial rows: {df.count()}")
log("Silver", f"Columns: {df.columns}")

Schema Validation and selecting required columns

In [0]:
required_cols = ["ROW_ID","Order_ID","Order_Date","Ship_Date","Ship_Mode","Customer_ID","Customer_Name","Segment","Country","City","execution_datetime","source_file"]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.select(*required_cols)

log("Silver", "Schema validation passed")

In [0]:
df.display()

Renaming column

In [0]:
log("Silver", "Applying column rename map")

RENAME_MAP = {
    "ROW_ID": "row_id",
    "Order_ID": "order_id",
    "Order_Date": "order_date",
    "Ship_Date": "ship_date",
    "Ship_Mode": "ship_mode",
    "Customer_ID": "customer_id",
    "Customer_Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state"
}

df = df.select([
    col(c).alias(RENAME_MAP.get(c, c.lower()))
    for c in df.columns
])

log("Silver", f"Columns after rename: {df.columns}")

Type Standardization

In [0]:
log("Silver", "Converting date columns")

df = (
    df
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("ship_date", F.to_date("ship_date"))
)

Business Rule 1 : Valid Dates

In [0]:
log("Silver", "Applying business rule: valid shipment dates")

before_filter = df.count()

df = df.filter(
    (col("order_date").isNotNull()) &
    (col("ship_date").isNotNull()) &
    (col("ship_date") >= col("order_date"))
)

after_filter = df.count()
log("Silver", f"Rows after date validation: {after_filter} (removed {before_filter - after_filter})")

Business rule 2: Remove Duplicates

In [0]:
log("Silver", "Removing duplicates based on order_id")

before_dedup = df.count()
df = df.dropDuplicates(["order_id"])
after_dedup = df.count()

log("Silver", f"Rows after dedup: {after_dedup} (removed {before_dedup - after_dedup})")

Business Rule 3: Clean Strings

In [0]:
log("Silver", "Cleaning string columns")

df = df.select([
    F.trim(col(c)).alias(c) if dict(df.dtypes)[c] == "string" else col(c)
    for c in df.columns
])

Split Customer Name

In [0]:
if "customer_name" in df.columns:
    log("Silver", "Splitting customer_name")

    name_split = F.split(col("customer_name"), " ")

    df = (
        df
        .withColumn("customer_first_name", name_split[0])
        .withColumn("customer_last_name", name_split[F.size(name_split) - 1])
        .drop("customer_name")
    )

Partition order_date

In [0]:
log("Silver", "Adding order_date partition columns")

df = (
    df
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("day", F.dayofmonth("order_date"))
)

Final Row Count

In [0]:
log("Silver", f"Final rows before write: {df.count()}")

Writing to silver.orders

In [0]:
log("Silver", "Writing to silver.orders")

row_count = df.count()

if row_count == 0:
    raise ValueError("Silver dataset is empty - stopping pipeline")

try:
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .partitionBy("year", "month", "day") \
        .saveAsTable("silver.orders")

    log("Silver", f"Write completed successfully | rows_written= {row_count}")

except Exception as e:
    log("Silver", f"ERROR during write: {str(e)}")
    raise